In [ ]:
import sys
sys.path.insert(0, 'api')   # allows importing bocpd.py and hmm.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from bocpd import run_bocpd
from hmm import run_hmm

plt.style.use('seaborn-v0_8-darkgrid')

TICKER = "SPY"
START  = "2016-01-01"
END    = "2026-04-14"

In [ ]:
print("Running BOCPD...")
bocpd_result = run_bocpd(TICKER, START, END)

print("Running HMM...")
hmm_result = run_hmm(TICKER, START, END)

print(f"BOCPD changepoints: {bocpd_result['metadata']['n_changepoints']}")
print(f"HMM   changepoints: {hmm_result['metadata']['n_changepoints']}")

In [ ]:
prices_df = pd.DataFrame(bocpd_result["prices"]).set_index("date")

# Map dates to HMM labels
hmm_seq   = pd.DataFrame(hmm_result["state_sequence"]).set_index("date")
HMM_COLORS = {"bear": "#ef4444", "neutral": "#f59e0b", "bull": "#22c55e"}
BOCPD_COLOR = "#3b82f6"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

for ax, title in [(ax1, "BOCPD Regimes"), (ax2, "HMM 3-State Regimes")]:
    ax.plot(prices_df.index, prices_df["close"], color="#94a3b8", linewidth=0.7)
    ax.set_title(f"{TICKER} — {title}", fontsize=11)
    ax.set_ylabel("Price")

# Shade BOCPD changepoints as vertical lines
for cp in bocpd_result["changepoints"]:
    ax1.axvline(x=cp["date"], color=BOCPD_COLOR, alpha=0.5, linewidth=0.8)

# Shade HMM regime bands
state_labels = hmm_seq["label"].values
state_dates  = hmm_seq.index.tolist()
cur = state_labels[0]; seg_s = state_dates[0]
for i in range(1, len(state_labels)):
    if state_labels[i] != cur:
        ax2.axvspan(seg_s, state_dates[i], alpha=0.2, color=HMM_COLORS[cur])
        cur = state_labels[i]; seg_s = state_dates[i]
# Draw last segment
ax2.axvspan(seg_s, state_dates[-1], alpha=0.2, color=HMM_COLORS[cur])

patches = [mpatches.Patch(color=HMM_COLORS[l], alpha=0.5, label=l.capitalize())
           for l in ["bear", "neutral", "bull"]]
ax2.legend(handles=patches, loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Align both methods on same dates
hmm_seq_aligned = hmm_seq.reindex(prices_df.index)

# BOCPD regime label: derive from regime_segments
bocpd_segs = bocpd_result["regime_segments"]
bocpd_label_map = {}
for seg in bocpd_segs:
    for d in pd.date_range(seg["start"], seg["end"], freq="B"):
        date_str = str(d.date())
        mu = seg["mean_return_annual"]
        bocpd_label_map[date_str] = "bull" if mu > 0.05 else "bear" if mu < -0.05 else "neutral"

# Align
common_dates = [d for d in prices_df.index if d in bocpd_label_map and d in hmm_seq.index]
bocpd_labels = [bocpd_label_map[d] for d in common_dates]
hmm_labels   = [hmm_seq.loc[d, "label"] for d in common_dates]

agree = sum(b == h for b, h in zip(bocpd_labels, hmm_labels))
total = len(common_dates)
print(f"Day-level agreement: {agree}/{total} = {agree/total:.1%}")

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
label_order = ["bear", "neutral", "bull"]
cm = confusion_matrix(bocpd_labels, hmm_labels, labels=label_order)
disp = ConfusionMatrixDisplay(cm, display_labels=label_order)
disp.plot(cmap="Blues")
plt.title("BOCPD (rows) vs HMM (cols) regime label confusion")
plt.tight_layout()
plt.show()

In [ ]:
print("=== BOCPD Segments ===")
bocpd_df = pd.DataFrame(bocpd_result["regime_segments"])
print(bocpd_df[["id","start","end","n_days","mean_return_annual","std_annual"]].to_string(index=False))

print("\n=== HMM Segments ===")
hmm_df = pd.DataFrame(hmm_result["regime_segments"])
print(hmm_df[["id","label","start","end","n_days","mean_return_annual","std_annual"]].to_string(index=False))

print("\n=== HMM State Parameters ===")
params_df = pd.DataFrame(hmm_result["state_params"])
print(params_df.to_string(index=False))